# Leakage and atom loss in Clifft

Real devices leak (an atom is excited out of the qubit subspace) and lose atoms (an atom
leaves the trap). Ordinary Pauli noise cannot represent either. Clifft models both through
a *noncomputational* simulation path, shown below on small circuits -- each with a check
against a closed form -- and then on a distance-5 repetition code at published neutral-atom
magnitudes.

Coming from a transition-matrix leakage simulator such as cirq-superstaq? The final table
maps each of its concepts onto this API.

## The model

Every qubit carries five levels:

| index | name | category | meaning |
|---|---|---|---|
| 0 | `g` | computational | qubit $\lvert 0\rangle$ |
| 1 | `e` | computational | qubit $\lvert 1\rangle$ |
| 2 | `leak_g` | leaked | carrier present, outside the qubit subspace |
| 3 | `leak_e` | leaked | a second leaked level |
| 4 | `lost` | lost | carrier gone from the trap |

Per-gate **transition matrices** `T[to, from]` give the probability of jumping between
levels when a gate fires; a column's deficit below 1 is "no jump". A **classifier** maps
the level at readout to a recorded symbol, with a third symbol that heralds loss.
Occupation of the noncomputational levels is tracked as classical per-qubit status.

Clifft compiles once and, per shot, draws a classical loss/leakage trajectory, rewrites
the circuit for it, and runs the rewrite through the exact simulator. This is **exact**
when the computational transition rates are state-independent. When they are not, the draw
needs the live quantum state, so Clifft **refuses by default** -- and
`unknown_source_policy="exact"` moves those draws to sample time, exact for every source
context (shown below).

In [1]:
import matplotlib.pyplot as plt
import numpy as np

import clifft
from clifft import noncomp, twirl

Level = noncomp.Level
SHOTS = 20_000


# Build a 5x5 transition matrix T[to][from] from {(to, from): p}.
def T(entries: dict) -> list[list[float]]:
    m = [[0.0] * 5 for _ in range(5)]
    for (to, frm), p in entries.items():
        m[to][frm] = p
    return m


# Classifier: g/leak_g read 0, e/leak_e read 1 (with optional confusion), lost heralds.
def ternary_classifier(leak_confusion: float = 0.0) -> noncomp.Classifier:
    m = [[0.0] * 5 for _ in range(3)]
    m[0][Level.G] = m[1][Level.E] = 1.0
    m[0][Level.LEAK_G], m[1][Level.LEAK_G] = 1 - leak_confusion, leak_confusion
    m[1][Level.LEAK_E], m[0][Level.LEAK_E] = 1 - leak_confusion, leak_confusion
    m[2][Level.LOST] = 1.0
    return noncomp.Classifier(["0", "1", "2"], m)


print("clifft", clifft.version())

clifft 0.4.2.dev17+ge9f424da4.d20260611


## A model and its sidecars

A `noncomp.Model` is an initial level distribution, per-gate transition matrices, a
classifier, and policy knobs. `noncomp.sample` returns the usual records plus two sidecars:
each qubit's final status, and a per-measurement loss herald. `symbols()` folds the herald
back into a ternary (0 / 1 / loss) per-slot view.

In [2]:
model = noncomp.Model(
    initial_state=[0.95, 0.00, 0.02, 0.01, 0.02],  # P(g, e, leak_g, leak_e, lost)
    classifier=ternary_classifier(),
)
r = noncomp.sample("M 0\n", model, shots=SHOTS, seed=1)

status = r.final_status[:, 0]
print(
    "status fractions (computational, leaked, lost):",
    [round(float((status == k).mean()), 4) for k in range(3)],
)
print("expected:                                      ", [0.95, 0.03, 0.02])
print("ternary readout counts (0/1/2):", [int((r.symbols()[:, 0] == v).sum()) for v in (0, 1, 2)])
assert abs((status == noncomp.QubitStatusKind.LOST).mean() - 0.02) < 0.005

status fractions (computational, leaked, lost): [0.9513, 0.0297, 0.019]
expected:                                       [0.95, 0.03, 0.02]
ternary readout counts (0/1/2): [19430, 190, 380]


## Loss is a partial trace

When an entangled qubit is lost, the survivor is left in the reduced state. For a Bell pair
that is maximally mixed -- the survivor reads 0 or 1 with probability one half. The rewriter
implements this with a hidden collapse at the loss event; the visible record layout is
unchanged.

In [3]:
lossy = noncomp.Model(
    initial_state=[1, 0, 0, 0, 0],
    transitions={"S": T({(Level.LOST, Level.G): 1.0, (Level.LOST, Level.E): 1.0})},
    classifier=ternary_classifier(),
)
# Bell pair, then the hooked S loses qubit 0 with certainty.
r = noncomp.sample("H 0\nCX 0 1\nS 0\nM 0\nM 1\n", lossy, shots=SHOTS, seed=2)

survivor = r.measurements[:, 1]
print("survivor P(1) =", round(float(survivor.mean()), 4), " (partial trace: 0.5)")
print("lost qubit heralds:", bool(np.all(r.heralds[:, 0] == 1)))
assert abs(survivor.mean() - 0.5) < 0.01

survivor P(1) = 0.5037  (partial trace: 0.5)
lost qubit heralds: True


## Leakage becomes a detector event

A leaked or lost level is classified into the measurement record *before* detectors are
evaluated, so leakage surfaces as a detector event -- the whole point for error correction.
Here a leaked qubit always reads 1 against a reference that always reads 0, so the detector
always fires.

In [4]:
leaky = noncomp.Model(
    initial_state=[1, 0, 0, 0, 0],
    transitions={"S": T({(Level.LEAK_E, Level.G): 1.0, (Level.LEAK_E, Level.E): 1.0})},
    classifier=ternary_classifier(),  # leak_e reads 1
)
# A reference measurement, then the leaked one; the detector XORs them.
r = noncomp.sample("M 1\nH 0\nS 0\nM 0\nDETECTOR rec[-1] rec[-2]\n", leaky, shots=SHOTS, seed=4)
print(
    "P(detector fires) =",
    round(float(r.detectors[:, 0].mean()), 4),
    " (leaked qubit always reads 1, reference always 0)",
)
assert np.all(r.detectors[:, 0] == 1)

P(detector fires) = 1.0  (leaked qubit always reads 1, reference always 0)


## State-dependent rates: refuse, or resolve at runtime

If the leak rate differs between the two computational levels, the jump probability depends
on the amplitude and no ahead-of-time draw is exact, so Clifft refuses by default. Opting in
to `unknown_source_policy="exact"` moves the draw to sample time: the fire probability is
evaluated on the live state, so the leak fires only out of the amplitude actually in the
leaking level. On $\lvert +\rangle$, with a leak that reads 1, the marginal is
$\tfrac12 + \tfrac{p}{2}$.

In [ ]:
p = 0.4
leak_from_g = {"S": T({(Level.LEAK_E, Level.G): p})}  # leaks only out of g

strict = noncomp.Model(
    initial_state=[1, 0, 0, 0, 0], transitions=leak_from_g, classifier=ternary_classifier()
)
try:
    noncomp.sample("H 0\nS 0\nM 0\n", strict, shots=4, seed=5)
except ValueError as err:
    print("default policy refuses:", str(err)[:80], "...")

exact = noncomp.Model(
    initial_state=[1, 0, 0, 0, 0],
    transitions=leak_from_g,
    classifier=ternary_classifier(),
    unknown_source_policy="exact",
)
r = noncomp.sample("H 0\nS 0\nM 0\n", exact, shots=SHOTS, seed=5)
print(f"P(M=1) = {r.measurements[:, 0].mean():.4f}   (analytic: {0.5 + p / 2})")
leaked = (r.final_status[:, 0] == noncomp.QubitStatusKind.LEAKED).mean()
print(f"P(leaked) = {leaked:.4f}   (analytic: {p / 2})")
assert abs(r.measurements[:, 0].mean() - (0.5 + p / 2)) < 0.01

## What runtime resolution buys: correlations

Marginals like the one above cannot tell the sampling strategies apart -- any unbiased
source gives the same numbers. The difference is joint correlation: when the leak
destination depends on the source level, the exact draw conditions on the simulator's own
collapse, so the leaked atom's readout is correlated with its entangled partner. On a Bell
pair the two records agree on every shot -- a correlation ahead-of-time sampling
fundamentally cannot produce, pinned by a test in the repository.

In [ ]:
pair_model = noncomp.Model(
    initial_state=[1, 0, 0, 0, 0],
    transitions={"S": T({(Level.LEAK_G, Level.G): 1.0, (Level.LEAK_E, Level.E): 1.0})},
    classifier=ternary_classifier(),
    unknown_source_policy="exact",
)
r = noncomp.sample("H 0\nCX 0 1\nS 0\nM 0\nM 1\n", pair_model, shots=SHOTS, seed=6)
m = r.measurements
print("marginals:", round(float(m[:, 0].mean()), 3), round(float(m[:, 1].mean()), 3))
print("P(records equal) =", float((m[:, 0] == m[:, 1]).mean()))
assert (m[:, 0] == m[:, 1]).all()

## Downstream of a lost site

An operation on a leaked or lost operand with no representable effect is refused by default.
`lost_leaked_ops="drop"` excises such operations whole (identity on the surviving operands);
measurements are never dropped, so the record layout survives. With this and
`reset_restores_lost` on, multi-round circuits run through loss events end to end.

In [7]:
circuit = "H 0\nS 0\nCX 0 1\nMR 1\nCX 0 1\nMR 1\nM 0\n"
lose_data = {"S": T({(Level.LOST, Level.G): 1.0, (Level.LOST, Level.E): 1.0})}

strict = noncomp.Model(
    initial_state=[1, 0, 0, 0, 0], transitions=lose_data, classifier=ternary_classifier()
)
try:
    noncomp.sample(circuit, strict, shots=4, seed=8)
except ValueError as err:
    print("default policy refuses:", str(err)[:80], "...")

compat = noncomp.Model(
    initial_state=[1, 0, 0, 0, 0],
    transitions=lose_data,
    classifier=ternary_classifier(),
    lost_leaked_ops="drop",
)
r = noncomp.sample(circuit, compat, shots=8, seed=8)
print("ternary records (ancilla, ancilla, data):")
print(r.symbols()[:4])  # both CXs dropped: ancilla stays 0; the lost data heralds

default policy refuses: rewrite: operation 'CX' on a Lost qubit 0 at op 2 is not representable; rejectin ...
ternary records (ancilla, ancilla, data):
[[0 0 2]
 [0 0 2]
 [0 0 2]
 [0 0 2]]


## A realistic neutral-atom model

Parameter magnitudes follow Infleqtion's two-logical-qubit cesium experiment
([arXiv:2412.07670](https://arxiv.org/abs/2412.07670)): per-two-qubit-gate excited-state
leakage near $10^{-3}$ with a few $\times 10^{-3}$ atom loss, single-qubit-layer leakage near
$10^{-3}$, and a few percent readout confusion. The circuit is a distance-5 repetition-code
memory -- five rounds of CX syndrome extraction with measure-and-reset ancillas and detectors
comparing consecutive rounds. The detector and herald rates below are driven by leakage and
loss, and the policy knobs above let the whole thing run end to end.

In [ ]:
def rep_code(d: int, rounds: int, p_z: float) -> str:
    data = list(range(d))
    anc = list(range(d, 2 * d - 1))
    lines = ["H " + " ".join(map(str, data))]
    for k in range(rounds):
        for i in range(d - 1):
            lines += [f"CX {data[i]} {anc[i]}", f"CX {data[i + 1]} {anc[i]}"]
        lines += [
            "S " + " ".join(map(str, data)),
            f"Z_ERROR({p_z}) " + " ".join(map(str, data)),
            "MR " + " ".join(map(str, anc)),
        ]
        if k > 0:  # compare each ancilla with its previous round
            lines += [f"DETECTOR rec[-{i + 1}] rec[-{i + 1 + (d - 1)}]" for i in range(d - 1)]
    lines.append("M " + " ".join(map(str, data)))
    return "\n".join(lines) + "\n"


def atom_model(scale: float) -> noncomp.Model:
    leak_2q, loss_2q, leak_1q = 1.0e-3 * scale, 3.9e-3 * scale, 1.3e-3 * scale
    return noncomp.Model(
        initial_state=[1 - 0.0095 * scale, 0, 0.0065 * scale, 0.003 * scale, 0],
        transitions={
            "CX": T(
                {
                    (Level.LEAK_E, Level.E): leak_2q,
                    (Level.LOST, Level.E): loss_2q,
                    (Level.LEAK_G, Level.G): 1e-4 * scale,
                }
            ),
            "S": T({(Level.LEAK_E, Level.E): leak_1q}),
        },
        classifier=ternary_classifier(leak_confusion=0.028),
        reset_restores_lost=True,  # a fresh atom is loaded at ancilla reset
        unknown_source_policy="exact",
        lost_leaked_ops="drop",
    )


P_Z_OVERROT = twirl.rotation("Z", 0.012 * np.pi / 2)[2]
D, ROUNDS = 5, 5
text = rep_code(D, ROUNDS, P_Z_OVERROT)

scales = [0.0, 0.5, 1.0, 2.0, 4.0]
det_rate, herald_rate = [], []
for s in scales:
    r = noncomp.sample(text, atom_model(s), shots=4000, seed=10)
    det_rate.append(float(r.detectors.mean()))
    herald_rate.append(float(r.heralds.mean()))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2))
ax1.plot(scales, det_rate, "o-")
ax1.set_xlabel("noise scale (1.0 = published magnitudes)")
ax1.set_ylabel("detector event fraction")
ax2.plot(scales, herald_rate, "s-", color="tab:red")
ax2.set_xlabel("noise scale")
ax2.set_ylabel("loss-herald fraction")
fig.suptitle(f"d={D} repetition code, {ROUNDS} rounds")
fig.tight_layout()
plt.show()

## Exact, approximate, out of scope

**Exact** (within Clifft's exact near-Clifford simulation): initial level populations; loss
with partial-trace back-action and gates dropped at lost sites; leakage/loss transitions on
any state -- state-independent ones ahead of time, state-dependent ones at runtime under
`unknown_source_policy="exact"`, with the no-jump back-action applied exactly;
computational-destination transitions (relaxation, recapture); measurement classifiers,
confusion, the ternary loss herald, and detector wiring; Pauli/readout noise alongside all
of it; and coherent control errors simulated exactly.

**Approximate, opt-in, characterized:** `damping="neglect"` keeps the compiled rank flat by
omitting the no-jump back-action at coherent dormant sites -- a survivorship tilt of order
$|p_g - p_e|$, with fire-side correlations kept exact; coherent errors as twirled Pauli
channels (`clifft.twirl`) -- preserves the Pauli-transfer-matrix diagonal, not circuit
statistics.

**Out of scope:** movement / atom-site semantics and correlated multi-qubit leakage.

Anything Clifft cannot do exactly is refused by default and available only behind a named
knob whose approximation is documented and tested -- a simulation is either exact, or says
precisely how it is not.

## Heralded measurements and detectors

A `DETECTOR` is evaluated as the XOR of the visible record bits it references; it does not
see the `heralds` sidecar. A heralded (loss) measurement contributes a **uniformly drawn**
bit, so any detector touching a heralded slot becomes a fair coin -- a detector over two
heralded records is the XOR of two independent uniform bits, still 50/50, carrying no
syndrome information. This is deliberate: pinning the bit (say to 0) would feed the decoder
a fabricated, consistent-looking syndrome, whereas a uniform bit honestly reports that
nothing is known there.

The loss information is not thrown away -- it moves to the `heralds` sidecar, and the
visible record layout stays fixed so the detector slot and every `rec[-k]` reference
survive. A leakage/erasure-aware decoder reads the heralds and treats those detectors as
**erasures** (known-location errors, which is exactly where loss decoding gains its
advantage); a herald-blind decoder simply sees random syndrome noise on those checks, which
correctly degrades the logical error rate rather than silently hiding the loss.

## Translating from cirq-superstaq's leakage simulator

| there | here |
|---|---|
| five-level qudit, indices 0--4 | the same five levels, `noncomp.Level` |
| `JumpChannel` matrix (column-to-row prob, deficit = no jump) | `transitions={"GATE": T}`, identical convention |
| channels on CZ / Z-rotations | a matrix on any gate name (`"CZ"`, `"S"`, `"CX"`, ...) |
| initial-state preparation channel | `initial_state=[...]` |
| ternary readout (0 / 1 / loss) | three-symbol `Classifier`; the record stays binary, loss arrives in `heralds`, `symbols()` reconstructs the ternary view |
| equalize-and-collapse on superposed qubits | superseded: `unknown_source_policy="exact"` resolves these draws at runtime, no collapse approximation |
| gates skipped on leaked or lost atoms | `lost_leaked_ops="drop"` (opt-in) |
| reset reloads a lost atom | `reset_restores_lost=True` |
| over-rotations twirled onto the sampler | `clifft.twirl` (or simulate the rotation exactly) |
| `cirq.Circuit` input | Stim-format text; `DETECTOR` / `OBSERVABLE_INCLUDE` evaluated in-circuit |

`atom_model` above is this translation applied to published magnitudes. **Not available
here:** oversampling amortization and atom-movement noise hooks. **Here without a counterpart there:** in-circuit detectors and observables
computed from classifier outcomes, a record layout invariant under every loss event,
final-status and herald sidecars, exact non-Clifford and coherent-gate simulation, and
reproducible per-shot seeding.

#### Computational readout confusion ports directly

cirq-superstaq's classifier confuses the computational levels too -- its `g` / `e` columns
carry (asymmetric) readout error. Clifft honors those columns: readout error is purely
classical (a misreported bit, not a disturbance of the quantum state), so the columns become
an asymmetric flip of the recorded bit, applied in-circuit so detectors and feedback see the
flipped bit. A ported classifier keeps its computational readout error with no translation
step.